<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/Subtask_(B)_Hate_Speech_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_en = pd.read_csv('/content/StereoQueerEval_EN_training.tsv',sep='\t')
df_en.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_EN_0001,[CHANNEL]: Lesbian denied communion at funeral,[CHANNEL] speaks to Barbara Johnson who says a...,She's living in unrepentant defiance to God's ...,yes,yes_explicit,individual_l
1,training_EN_0002,GOP Congresswoman uses anti-trans slur in Hous...,Rep. Nancy Mace (R-SC) was challenged by Rep. ...,Rep Mace: Keep on being dedicated to truth and...,yes,yes_implicit,group_t
2,training_EN_0003,Kevin Smith defends his gay brother,"Kevin Smith supports gay rights, and defends h...","We all need Jesus, He said no homosexual shall...",yes,yes_implicit,"group_l,g"
3,training_EN_0004,Judge removes child from gay foster parents,A Utah judge removed a foster child from a les...,So can two black men adopt a little white girl...,yes,yes_implicit,group_l
4,training_EN_0005,Chechen leader Ramzan Kadyrov questioned on 'g...,Chechnya's leader Ramzan Kadyrov is questioned...,Of course west and US will hate him,yes,yes_implicit,group_g


### EXperiment A
comment ony as input

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:
df = df_en

In [6]:
df['hate_speech'].value_counts()

,count
hate_speech,
no,1407
yes_explicit,810
yes_implicit,772


In [7]:
df.isnull().sum()

,0
StereoQueerEval_id,0
yt_title,0
yt_description,0
yt_comment,0
stereotype,0
hate_speech,0
target,0


In [107]:
X = list(df['yt_comment'])
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2}).tolist()

In [108]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [10]:
!pip install transformers

### sentiment-roberta-large-english

In [11]:
from transformers import AutoTokenizer

MODEL = f"siebert/sentiment-roberta-large-english"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [52]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [53]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset = TextDataset(test_encodings, y_test)

In [45]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [46]:
from transformers import AutoModelForSequenceClassification, Trainer

# Explicitly set num_labels=3 to match our targets (0, 1, 2) and ignore mismatched classification head sizes
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True
).to(device)

trainer_RoBERTa = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_RoBERTa.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
10,1.232866
20,1.154900
30,1.091446
40,1.150599
50,1.023133
60,1.071005
70,1.097961
80,1.040664
90,1.113951
100,1.061719


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=1.0141036796569824, metrics={'train_runtime': 407.5908, 'train_samples_per_second': 23.465, 'train_steps_per_second': 1.472, 'total_flos': 629104181787648.0, 'train_loss': 1.0141036796569824, 'epoch': 4.0})

In [54]:
trainer_RoBERTa.evaluate(val_dataset)

Training Loss,Validation Loss,Step
0.934224,1.112486,600


{'eval_loss': 1.1124863624572754}

In [55]:
y_pred = np.argmax(trainer_RoBERTa.predict(test_dataset).predictions, axis=-1)

In [56]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.55      0.76      0.64       142
           1       0.44      0.33      0.38        81
           2       0.25      0.13      0.17        76

    accuracy                           0.48       299
   macro avg       0.41      0.41      0.40       299
weighted avg       0.44      0.48      0.45       299



###BERT

In [105]:
from transformers import  AutoTokenizer

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [109]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [110]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset = TextDataset(test_encodings, y_test)

In [113]:
from transformers import AutoModelForSequenceClassification,Trainer,TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

trainer_BERT = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer_BERT.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
10,1.824650
20,1.370099
30,1.128787
40,1.104309
50,1.098619
60,1.047865
70,1.183939
80,1.067232
90,1.009289
100,0.996264


In [72]:
from accelerate.state import PartialState
from transformers import Trainer

# Reset and properly re-instantiate the global accelerator state
PartialState._reset_state()
_ = PartialState()

# Re-create the trainer so it uses the clean accelerator state
trainer_BERT = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_BERT.evaluate(val_dataset)

Training Loss,Validation Loss,Step
No log,1.112486,0


{'eval_loss': 1.1124862432479858}

In [73]:
y_pred = np.argmax(trainer_BERT.predict(test_dataset).predictions, axis=-1)

In [74]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.55      0.76      0.64       142
           1       0.44      0.33      0.38        81
           2       0.25      0.13      0.17        76

    accuracy                           0.48       299
   macro avg       0.41      0.41      0.40       299
weighted avg       0.44      0.48      0.45       299



###cardiffnlp/twitter-xlm-roberta

In [24]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [75]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [76]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset = TextDataset(test_encodings, y_test)

In [32]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10
)

In [33]:
from transformers import AutoModelForSequenceClassification, Trainer


model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)

trainer_XLMRoBERTa = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_XLMRoBERTa.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.11GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Step,Training Loss
10,1.625623
20,1.284004
30,1.107147
40,1.079200
50,1.076611
60,1.026035
70,1.132416
80,1.037496
90,0.992830
100,1.034906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.8626835569490159, metrics={'train_runtime': 446.6518, 'train_samples_per_second': 10.706, 'train_steps_per_second': 1.339, 'total_flos': 314552090893824.0, 'train_loss': 0.8626835569490159, 'epoch': 2.0})

In [77]:
trainer_XLMRoBERTa.evaluate(val_dataset)

Training Loss,Validation Loss,Step
0.675057,1.941385,598


{'eval_loss': 1.94138503074646}

In [78]:
y_pred = np.argmax(trainer_XLMRoBERTa.predict(test_dataset).predictions, axis=-1)

In [79]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      0.01      0.01       142
           1       0.21      0.07      0.11        81
           2       0.26      0.91      0.40        76

    accuracy                           0.25       299
   macro avg       0.49      0.33      0.17       299
weighted avg       0.60      0.25      0.14       299



### DistilBERT

In [37]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [80]:
train_encodings = tokenizer(X_train, truncation= True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation= True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation= True, padding=True, max_length=128)

In [81]:
import torch

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset = TextDataset(test_encodings, y_test)

In [40]:
from transformers import (
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=60,
    weight_decay=0.01,
    logging_steps=10,
)

In [41]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english", num_labels=3, ignore_mismatched_sizes=True).to(device)

trainer_distilBERT = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer_distilBERT.train()

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Step,Training Loss
10,1.061266
20,1.055131
30,1.073817
40,1.077763
50,1.067053
60,1.031613
70,1.139366
80,1.019462
90,0.981449
100,0.925892


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=598, training_loss=0.8509343595409075, metrics={'train_runtime': 118.3651, 'train_samples_per_second': 40.4, 'train_steps_per_second': 5.052, 'total_flos': 158367599304192.0, 'train_loss': 0.8509343595409075, 'epoch': 2.0})

In [82]:
trainer_distilBERT.evaluate(val_dataset)

Training Loss,Validation Loss,Step
0.751573,0.907526,598


{'eval_loss': 0.907526433467865}

In [83]:
y_pred = np.argmax(trainer_distilBERT.predict(test_dataset).predictions, axis=-1)

In [84]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.68      0.68      0.68       142
           1       0.51      0.47      0.49        81
           2       0.32      0.34      0.33        76

    accuracy                           0.54       299
   macro avg       0.50      0.50      0.50       299
weighted avg       0.54      0.54      0.54       299



### EXperiment
comment + title as input

In [86]:
df.columns

Index(['StereoQueerEval_id', 'yt_title', 'yt_description', 'yt_comment',
       'stereotype', 'hate_speech', 'target'],
      dtype='object')

In [89]:
X = (df.drop(['StereoQueerEval_id','yt_description','stereotype','target'],axis=1))
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2})

In [90]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [93]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    ctx = title
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [95]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [96]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [101]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [100]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [103]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.069120
20,1.063781
30,1.019820
40,1.032028
50,0.968369
60,0.996223
70,0.995357
80,0.918979
90,0.932613
100,0.982322


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.6914786970615387, metrics={'train_runtime': 321.2615, 'train_samples_per_second': 29.77, 'train_steps_per_second': 1.868, 'total_flos': 790767849032532.0, 'train_loss': 0.6914786970615387, 'epoch': 4.0})

In [104]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.714     0.775     0.743       142
    explicit      0.641     0.617     0.629        81
    implicit      0.448     0.395     0.420        76

    accuracy                          0.635       299
   macro avg      0.601     0.596     0.597       299
weighted avg      0.627     0.635     0.630       299

[[110  15  17]
 [ 11  50  20]
 [ 33  13  30]]
